In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\full_dataset\with_snomed_category.csv"
# df_path = r"D:\DATA\EXP3_abmil\abmil_inference.csv"
# df_path = r"D:\DATA\EXP3_abmil\abmil_training.csv"

df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to cache file (tissue artifact feature cache)
cache_path = r"D:\DATA\cache_tissue_artifact_features.pkl"

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
all_filenames = df_all["filename"].tolist()
print("Number of files: ", len(all_filenames))

In [ ]:
from helper_functions import subset_df, subset_df_list, subset_df_word, subset_df_processed

df_HE = subset_df(df_all, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
df_HE = df_HE[df_HE['T_category'].apply(len) == 1]

In [ ]:
from tissue_artifact_segmentation import SegmentMany

segmenter = SegmentMany(all_filenames, cache_path, zarr_dir, "tissue", version="default")

In [ ]:
# Visualize Single Slide
import os
from wsidata import open_wsi

path = all_filenames[2]
print(path)

zarr_path = os.path.join(zarr_dir, os.path.basename(path).replace(".mrxs", ".zarr"))
wsi = open_wsi(path, zarr_path)
wsi

In [ ]:
import lazyslide as zs
viewer = zs.pl.WSIViewer(wsi)
viewer.add_image()
viewer.add_contours(key = 'tissue_default')
viewer.show()